# מחברת 1: יסודות CUDA על A100

מחברת זו מתחילה ב-thread יחיד ומוסיפה רעיון אחד בכל שלב. אל תריצו Run All בפעם הראשונה; קראו, נחשו, ורק אז הריצו כל תא.

In [ ]:
from pathlib import Path
import shutil
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
required = ("nvidia-smi", "nvcc", "cmake")
missing = [command for command in required if shutil.which(command) is None]
assert not missing, f"Missing required commands: {missing}"
print(subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True, check=True).stdout)


## Build

הבנייה מכוונת ל-`sm_80`. התא בונה רק את מסלול היסודות.

In [ ]:
subprocess.run([
    "cmake", "-S", str(ROOT), "-B", str(ROOT / "build"),
    "-DCMAKE_BUILD_TYPE=Release", "-DCMAKE_CUDA_ARCHITECTURES=80"
], check=True)
targets = [
    "00_device_query", "01_hello_kernel", "02_one_block_index",
    "03_global_index", "04_bounds_check", "05_memory_roundtrip", "06_vector_add",
]
subprocess.run(["cmake", "--build", str(ROOT / "build"), "--target", *targets, "-j"], check=True)


## שלב 1: CPU מפעיל kernel על GPU

ניחוש לפני הרצה: כמה פעמים יודפס `Hello from the GPU`?

In [ ]:
result = subprocess.run([str(ROOT / "build/01_hello_kernel")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS hello_kernel" in result.stdout


## שלב 2: ארבעה threads בתוך block אחד

ניחוש: מהם ארבעת ערכי `threadIdx.x`? סדר ההדפסה יכול להשתנות.

In [ ]:
result = subprocess.run([str(ROOT / "build/02_one_block_index")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS one_block_index" in result.stdout


## שלב 3: שני blocks ואינדקס גלובלי

הנוסחה היא `blockIdx.x * blockDim.x + threadIdx.x`. חשבו את האינדקס של thread 2 ב-block 1 לפני ההרצה.

In [ ]:
result = subprocess.run([str(ROOT / "build/03_global_index")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS global_index" in result.stdout


## שלב 4: שישה ערכים, שמונה threads

שני threads עודפים. ה-guard מונע מהם לגעת בזיכרון שאינו שייך למערך.

In [ ]:
result = subprocess.run([str(ROOT / "build/04_bounds_check")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS bounds_check" in result.stdout


## שלב 5: זיכרון CPU ו-GPU

ה-vector מתחיל ב-CPU, מועתק ל-GPU, משתנה ב-kernel וחוזר ל-CPU.

In [ ]:
result = subprocess.run([str(ROOT / "build/05_memory_roundtrip")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS memory_roundtrip" in result.stdout


## שלב 6: Vector add ו-residual connection

כל thread מחבר איבר אחד. ב-LLM זה אותו pattern של `hidden = layer_output + residual`.

In [ ]:
result = subprocess.run([str(ROOT / "build/06_vector_add")], text=True, capture_output=True)
print(result.stdout, result.stderr)
assert result.returncode == 0 and "PASS vector_add" in result.stdout


## בדיקת סיום

הסבירו בקול:

1. מה ההבדל בין block ל-thread?
2. למה האינדקס מתחיל ב-0?
3. למה יש threads עודפים?
4. איפה הנתונים נמצאים לפני ואחרי `cudaMemcpy`?
5. איך vector addition קשור ל-residual connection?